In [1]:
2+3

5

In [5]:
import os
import cv2
import torch
import timm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
from sklearn.metrics import accuracy_score

In [8]:
DATA_DIR = "dataset/train"   # your dataset folder
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [9]:
class AIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = []
        self.labels = []

        for label, folder in enumerate(["real", "fake"]):
            folder_path = os.path.join(root_dir, folder)

            for img in os.listdir(folder_path):
                self.image_paths.append(os.path.join(folder_path, img))
                self.labels.append(label)

        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[idx]).float()
        return image, label

In [10]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [13]:
train_dataset = AIDataset("../dataset/train", transform=train_transform)
val_dataset = AIDataset("../dataset/val", transform=val_transform)
test_dataset = AIDataset("../dataset/test", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 12600
Val: 2700
Test: 2700


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model("efficientnet_b3", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 1)

model = model.to(device)

In [16]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [17]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ===== VALIDATION =====
    model.eval()
    preds = []
    true = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)

            outputs = torch.sigmoid(model(images)).cpu()
            preds.extend((outputs > 0.5).int().numpy())
            true.extend(labels.numpy())

    acc = accuracy_score(true, preds)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Accuracy: {acc:.4f}")

Epoch 1
Train Loss: 143.8894
Val Accuracy: 0.9685
Epoch 2
Train Loss: 49.3309
Val Accuracy: 0.9833
Epoch 3
Train Loss: 30.7264
Val Accuracy: 0.9752
Epoch 4
Train Loss: 20.5309
Val Accuracy: 0.9741
Epoch 5
Train Loss: 13.9392
Val Accuracy: 0.9741
Epoch 6
Train Loss: 14.5931
Val Accuracy: 0.9793
Epoch 7
Train Loss: 10.1891
Val Accuracy: 0.9789
Epoch 8
Train Loss: 10.6702
Val Accuracy: 0.9837
Epoch 9
Train Loss: 7.1464
Val Accuracy: 0.9841
Epoch 10
Train Loss: 12.8310
Val Accuracy: 0.9804


In [18]:
model.eval()
preds = []
true = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = torch.sigmoid(model(images)).cpu()
        preds.extend((outputs > 0.5).int().numpy())
        true.extend(labels.numpy())

test_acc = accuracy_score(true, preds)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.9781481481481481


In [19]:
torch.save(model.state_dict(), "ai_detector.pth")
print("Model saved!")

Model saved!


In [7]:
for split in ["train", "val", "test"]:
    real = len(os.listdir(f"../dataset/{split}/real"))
    fake = len(os.listdir(f"../dataset/{split}/fake"))
    
    print(f"{split.upper()} → Real: {real}, Fake: {fake}")

TRAIN → Real: 6300, Fake: 6300
VAL → Real: 1350, Fake: 1350
TEST → Real: 1350, Fake: 1350


In [20]:
del model
del images
